In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Install and Import Libraries**

In [1]:
!pip install -q transformers datasets accelerate evaluate wandb sentencepiece scikit-learn

import os
import gc
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    DataCollatorForMultipleChoice
)

from datasets import Dataset as HFDataset

import evaluate

import wandb

from tqdm.auto import tqdm

print(torch.__version__)
print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 72.5 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cud

# **Load Dataset**

In [6]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

label2id={"A":0,"B":1,"C":2,"D":3,"E":4}
id2label={0:"A",1:"B",2:"C",3:"D",4:"E"}

train["label"]=train.answer.map(label2id)

train_df,valid_df=train_test_split(
    train,
    test_size=0.1,
    stratify=train.label,
    random_state=42
)

# **Label Encoding**

In [7]:
MODEL_NAME="microsoft/deberta-v3-base"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN=256

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

# **Train Validation Split**

In [9]:
class MCQDataset(Dataset):

    def __init__(self,df,is_test=False):

        self.df=df.reset_index(drop=True)
        self.is_test=is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        question=row["prompt"]

        options=[
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encoding=tokenizer(
            [question]*5,
            options,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item={
            "input_ids":encoding["input_ids"],
            "attention_mask":encoding["attention_mask"]
        }

        if not self.is_test:
            item["labels"]=torch.tensor(row["label"])

        return item

# **Tokenizer**

In [10]:
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# **Dataset Class**

In [11]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len=256, is_test=False):

        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        question = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"]
        }

        if not self.is_test:
            item["labels"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# **DataLoader**

In [12]:
train_dataset = MCQDataset(
    train_df,
    tokenizer,
    MAX_LEN
)

valid_dataset = MCQDataset(
    valid_df,
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8,
    shuffle=False
)

print("Train batches :", len(train_loader))
print("Validation batches :", len(valid_loader))

Train batches : 225
Validation batches : 25


# **Load DeBERTa**

In [15]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    "microsoft/deberta-v3-base"
)

model.to(device)

print("Model Loaded Successfully")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                

Model Loaded Successfully


# **Optimizer + Scheduler**

In [16]:
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 3
LR = 2e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(total_steps)

675


# **Training Function**

In [18]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0

    progress = tqdm(loader)

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        progress.set_postfix(loss=loss.item())

    return total_loss / len(loader)

# **Training and Validation Functions**

In [22]:
from sklearn.metrics import accuracy_score, f1_score

def map3_score(y_true, logits):
    """
    Compute MAP@3 for single-label classification.
    """
    top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

    score = 0.0

    for i in range(len(y_true)):
        if y_true[i] in top3[i]:
            rank = np.where(top3[i] == y_true[i])[0][0] + 1
            score += 1 / rank

    return score / len(y_true)


def train_one_epoch(model, loader, optimizer, scheduler):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# **Validation Function**

In [23]:
def validate(model, loader):

    model.eval()

    predictions = []
    labels_list = []
    logits_list = []

    with torch.no_grad():

        for batch in tqdm(loader):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            logits_list.extend(logits.cpu().numpy())

    accuracy = accuracy_score(labels_list, predictions)

    f1 = f1_score(
        labels_list,
        predictions,
        average="macro"
    )

    map3 = map3_score(
        np.array(labels_list),
        np.array(logits_list)
    )

    return accuracy, f1, map3

# **Model Training**

In [24]:
best_map3 = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler
    )

    accuracy, f1, map3 = validate(
        model,
        valid_loader
    )

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Accuracy   : {accuracy:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print(f"MAP@3      : {map3:.4f}")

    if map3 > best_map3:

        best_map3 = map3

        torch.save(
            model.state_dict(),
            "best_deberta_mcq.pth"
        )

        print("Best model saved.")


Epoch 1/3


  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Train Loss : nan
Accuracy   : 0.1850
F1 Score   : 0.0624
MAP@3      : 0.3267
Best model saved.

Epoch 2/3


  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Train Loss : nan
Accuracy   : 0.1850
F1 Score   : 0.0624
MAP@3      : 0.3267

Epoch 3/3


  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Train Loss : nan
Accuracy   : 0.1850
F1 Score   : 0.0624
MAP@3      : 0.3267


# **Inference on Test Dataset**

In [25]:
# Create Test Dataset
test_dataset = MCQDataset(
    test,
    tokenizer,
    MAX_LEN,
    is_test=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

# Load Best Model
model.load_state_dict(torch.load("best_deberta_mcq.pth"))
model.eval()

predictions = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        top3 = torch.topk(logits, k=3, dim=1).indices

        predictions.extend(top3.cpu().numpy())

  0%|          | 0/63 [00:00<?, ?it/s]

# **Generate Submission File**

In [26]:
label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = [
    " ".join(label_map[i] for i in pred)
    for pred in predictions
]

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,ID,Prediction
0,1,C B A
1,2,C B A
2,3,C B A
3,4,C B A
4,5,C B A
